# Pre-processing, processing and cell type annotation of Xenium data

Spatial-data analysis for the Xenium dataset.


## Setup

Imports and dependencies used throughout the analysis.


In [ ]:
import h5py
import warnings
import os
import spatialdata_plot
from pathlib import Path
import logging
from matplotlib.patches import Patch
import spatialdata_io as sio


## Spatial preprocessing

Prepare coordinate information and spatial metadata.


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
import spatialdata as sd


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:

# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Set output directory for scanpy figures
sc.settings.figdir = "../Figures_forpaper/noWT3spatial"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Set output directory for the rest of the figures
out_dir = Path("../Figures_forpaper/noWT3spatial")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


## Generation of SpatialData objects

Xenium output for each sample was converted to a `SpatialData`
object using `spatialdata_io.xenium`.

The `morphology_focus` images were removed because they were not required for
the downstream analyses and substantially increased the size of the stored
objects.

Each sample was then saved as an individual `.zarr` file. These files were used
as input for the combined spatial analysis below.

The same processing was applied to all 12 samples: four WT samples
(`WT_rep1`, `WT_rep2`, `WT_rep4`, `WT_rep5`), four BE samples
(`BE_rep1`–`BE_rep4`), and four PBS samples (`PBS_rep1`–`PBS_rep4`).


In [ ]:
# Samples included in the spatial analysis
sample_ids = [
    "WT_rep1",
    "WT_rep2",
    "WT_rep4",
    "WT_rep5",
    "BE_rep1",
    "BE_rep2",
    "BE_rep3",
    "BE_rep4",
    "PBS_rep1",
    "PBS_rep2",
    "PBS_rep3",
    "PBS_rep4",
]

# Compute and store `input_dir`.
input_dir = Path("largecell_v4_segmentation")
# Compute and store `output_dir`.
output_dir = Path("Zarrfiles/Large_seg")
# Run `output_dir.mkdir` for this analysis step.
output_dir.mkdir(parents=True, exist_ok=True)

# Repeat the following operation for each item in the selected collection.
for sample_id in sample_ids:
    print(f"Processing {sample_id}...")

    xenium_path = (
        input_dir
        / f"{sample_id}_resegment_largeCell"
        / "outs"
    )

    # Read the resegmented Xenium output as a SpatialData object.
    sdata = sio.xenium(xenium_path)

    # Morphology images are not required for the downstream analyses.
    if "morphology_focus" in sdata:
        del sdata["morphology_focus"]

    # Store each sample as an individual Zarr-backed SpatialData object.
    output_path = output_dir / f"{sample_id}.zarr"
    sdata.write(output_path)

    print(f"Saved {sample_id} to {output_path}")


In [ ]:
#Uploading a metadata .csv file
meta_samples = pd.read_csv("sample_ids_names_largecells_zarr.csv", sep = ';')


In [ ]:
# Inspect the first rows of the resulting table.
meta_samples.head()


In [ ]:
# Define the values used for `adatas`.
adatas = []
# Repeat the following operation for each item in the selected collection.
for _, row in meta_samples.iterrows():
    sample = row["Sample_ID"]
    path   = row["Path"]

    sdata = sd.read_zarr(path)
    adata = sdata["table"]

    # keep original cell IDs
    adata.obs["cell_id_orig"] = adata.obs["cell_id"]
    adata.obs["id"] = sample

    # add sample-level info from CSV (e.g. sample_type)
    adata.obs["type"] = row["Sample_type"]
    adata.obs["name"] = row["Sample_name"]

    # make obs_names unique
    adata.obs_names = [f"{sample}_{cid}" for cid in adata.obs["cell_id_orig"]]

    adatas.append(adata)

# Concatenate samples into the `adata_combined`.
adata_combined = ad.concat(adatas, axis=0, index_unique=None)


In [ ]:
# list all column names
print(adata_combined.obs.columns)


In [ ]:
# Run `print` for this analysis step.
print(adata_combined.obsm.keys())


In [ ]:
# Run `print` for this analysis step.
print(adata_combined.obsm["spatial"][:5])


In [ ]:
# quick peek at the first few rows
adata_combined.obs.head()


In [ ]:
# slightly more detailed info
adata_combined.obs.info()


In [ ]:
# Inspect the number of observations in each category.
adata_combined.obs.id.value_counts()


In [ ]:
# Save the processed object
adata_combined.write_h5ad("largecells_all_withcoordinates.h5ad")


## Calculate QCs, filter cells and genes, create "counts" layer

In [ ]:
#the default percent_top will give Indexerror bc it goes to 500 genes. Basically this thing calculates what is the percentage of the counts taken by the selected number of genes - then the cells would be problematic ans have low complexity if eg 10 most expressed genes occupy 90% of counts
sc.pp.calculate_qc_metrics(combined_adata, percent_top=(10, 20, 50, 150, 200), inplace=True, log1p=True)

In [ ]:
#Visualise the calculated QC metrics
sc.pl.violin(
    combined_adata,
    ["n_genes_by_counts", "total_counts"],
    jitter=0.1, #default was 0.4 but gets too dense, cant see anything
    groupby="name",
    multi_panel=True,
)

In [ ]:
fig = plt.hist(combined_adata.obs["total_counts"], range=(0, 2000), bins=100)
plt.axvline(x=3, color="r", linestyle="--")

In [ ]:
sc.pl.violin(combined_adata, keys='pct_counts_in_top_50_genes', stripplot=False)

In [ ]:
# We store the counts in the layers in case we need it for future purposes
combined_adata.layers["counts"] = combined_adata.X.copy()

In [ ]:
#Filter the cells and genes
sc.pp.filter_cells(combined_adata, min_counts=50)
sc.pp.filter_genes(combined_adata, min_cells=3)

In [ ]:
# Again, we store the counts layer in the anndata object
combined_adata.layers["counts"] = combined_adata.X.copy()

In [ ]:
#Save the h5ad file
combined_adata.write_h5ad("combined_Xenium_largecells_filtered.h5ad")

## Load the fitrered data

Load spatial coordinates, annotations, and other analysis inputs.


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad("combined_Xenium_largecells_filtered.h5ad")


In [ ]:
# Check the unqiue names present
combined_adata.obs['name'].unique()


In [ ]:
# Remove WT_rep3 sample (due to tthe wrong sample section orientation)
combined_adata = combined_adata[combined_adata.obs['name'] != 'WT_rep3'].copy()


## Normalise, log-tranform, scale the data and run PCA

In [ ]:
# Find HVGs with seurat v3 flavour (expects counts data)
sc.pp.highly_variable_genes(combined_adata, flavor="seurat_v3", n_top_genes=200, layer="counts")


In [ ]:
# Normalise raw counts
sc.pp.normalize_total(combined_adata)
# logarithmise the counts
sc.pp.log1p(combined_adata)
# Make an independent copy of the selected data.
combined_adata.layers["lognorm"] = combined_adata.X.copy()


In [ ]:
# Scale the data (without zero centering due to the size of the dataset)
sc.pp.scale(combined_adata, zero_center=False, max_value=10)
# Run PCA
sc.pp.pca(combined_adata, n_comps=30)


In [ ]:
# Check how many PCs to use
sc.pl.pca_variance_ratio(combined_adata, n_pcs=30, log=True)


## UMAP embedding and leiden clustering


In [ ]:
# Compute the nearest neighbors distance matrix and a neighborhood graph of observations
sc.pp.neighbors(combined_adata, n_neighbors = 100, n_pcs = 25)
# Compute UMAP embedding
sc.tl.umap(combined_adata, min_dist = 0.05, spread = 2)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color="type")


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad("combined_Xenium_largecells_noWT3_afterclustering.h5ad")


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad("combined_Xenium_largecells_noWT3_afterclustering.h5ad")


In [ ]:
# Run `sc.tl.leiden` for this analysis step.
sc.tl.leiden(combined_adata, flavor="igraph", n_iterations=-1, resolution=1, key_added="leiden_r1")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color="leiden_r1")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color="F830016B08Rik")


## Clusters annotation

Annotate clusters based on the top markers expressed


In [ ]:
# Find differentially expressed genes for each of the leiden clusters
sc.tl.rank_genes_groups(combined_adata, groupby = "leiden_r1", method = "wilcoxon")


In [ ]:
# Run `sc.tl.dendrogram` for this analysis step.
sc.tl.dendrogram(combined_adata, groupby='leiden_r1')


In [ ]:
# Visualise the top 10 differentially expressed genes per leiden cluster with a dotplot
sc.pl.rank_genes_groups_dotplot(combined_adata, groupby = "leiden_r1", standard_scale="var", n_genes=10)


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden.h5ad")


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden.h5ad")


In [ ]:
# Map Leiden cluster IDs to cell type labels 
cluster_map = {
    "0": "Ifgga4+ VCMs",
    "1": "VCMs",
    "2": "VCMs",
    "3": "VCMs", #Myh7b positive also
    "4": "Stressed VCMs", #weird shape, probably need to cleanup
    "5": "Vasculature ECs",
    "6": "Myeloid", 
    "7": "FBs",
    "8": "Stressed VCMs",
    "9": "VCMs",
    "10": "FBs",
    "11": "Myh7+ VCMs",
    "12": "NCs",
    "13": "VCMs",
    "14": "Pericytes",
    "15": "Stressed VCMs",
    "16": "SMCs",
    "17": "Lymphoid",
    "18": "Vasculature ECs",
    "19": "Endocardial ECs",
    "20": "Vasculature ECs", 
    "21": "Epicardium",
    "22": "FBs",
    "23": "ACMs"
}

# Compute and store cell types in `combined_adata.obs['celltype']`.
combined_adata.obs["celltype"] = combined_adata.obs["leiden_r1"].map(cluster_map)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color="celltype")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color="Myh7")


## Visualization

Visualize spatial distributions and derived results.


In [ ]:
# Compute and store `(fig, axes)`.
fig,axes=plt.subplots(1,3,figsize=(20,7))
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300)

# Plot the UMAP embedding per sample type using the selected annotation or feature.
sc.pl.umap(combined_adata[combined_adata.obs["type"] == "R636Q"], 
           color = "leiden_r1", size = 2, ax=axes[0],title = "Mutant",legend_loc=None,show=False, frameon=False)
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata[combined_adata.obs["type"] == "WT"], 
           color = "leiden_r1", size = 2, ax=axes[1],title = "WT",legend_loc=None,show=False, frameon=False)
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata[combined_adata.obs["type"] == "BE"], 
           color = "leiden_r1", size = 2, ax=axes[2],title = "Base-edited",legend_loc=None,show=False, frameon=False)


# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Display the completed figure.
plt.show()


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad('combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated.h5ad')


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad('combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated.h5ad')


In [ ]:
# Define the desired order of cell types
ordered_cell_types = ['VCMs', 'Ifgga4+ VCMs','Stressed VCMs', 'Myh7+ VCMs', 'FBs', 'Vasculature ECs', 'Endocardial ECs', 'Epicardium', 'Myeloid', 'Lymphoid', 'Pericytes', 'SMCs', 'NCs', 'ACMs']


In [ ]:
# Assign the desired order of cell types
combined_adata.obs['celltype'] = pd.Categorical(combined_adata.obs['celltype'], categories=ordered_cell_types, ordered = True)


In [ ]:
# Color pallette
set1_14 = [
    "#6C5C8D", "#882255", "#24185F", "#DDCC77", "#712E00", "#88CCEE", "#97B1AB", "#44AA99", "#117733", "#999933",
    "#AA4499", "#7BB4C3", "#CC6677", "#332288"

]


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color="celltype", palette=set1_14, save = "UMAP_noWT3.pdf")


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad('combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered.h5ad')


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad('combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered.h5ad')


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color="celltype", palette=set1_14)


In [ ]:
# List of your marker genes of interest
marker_genes = ["Ttn", "F830016B08Rik", "Ankrd1", "Myh7", "Pdgfra", "Cdh5", "Vwf", "Muc16", "F13a1", "Skap1", "Rgs5", "Tagln", "Chl1", "Nppa"]
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300, dpi_save = 300, transparent = True)
# Plot the distribution of values between groups as a violin plot.
sc.pl.stacked_violin(
    combined_adata,
    marker_genes,
    groupby="celltype",
    swap_axes=False,      # optional: makes cell types on y-axis
    dendrogram=False,    # don’t cluster unless you want to
    figsize=(8,4),       # adjust as needed
    cmap="viridis",# colormap if continuous, but here we want by cell type
    save = "markers_violin.pdf"
)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color=["Cd74", "Bank1", "Ms4a1", "Cd247", "Skap1"])


In [ ]:
# Make an independent copy of the selected data.
LC = combined_adata[combined_adata.obs['celltype']=='Lymphoid'].copy()


In [ ]:
# Reload the counts data into the X to recluster the cells
LC.X = LC.layers["counts"].copy()


In [ ]:
# Run `sc.pp.highly_variable_genes` for this analysis step.
sc.pp.highly_variable_genes(LC, flavor="seurat_v3", layer="counts")


In [ ]:
# Exclude Rbm20 targets splicing isoforms as they should be mainly relevant for cardiomyocytes only
exclude_genes = [
    '1300002E11Rik-SE-neg', '2210408F21Rik-SE-pos', 'Aak1-SE-neg',
    'Abhd14a-SE-pos', 'Ank3-SE-pos', 'Arhgap10-MXE-neg', 'Arhgap10-MXE-pos',
    'Aste1-ASS-pos', 'Camk2d-SE-pos', 'Camk2d_flanking-exons',
    'Camk2d_isoformA', 'Camk2d_isoformB', 'Carnmt1-MXE-pos',
    'Ccdc125-MXE-neg', 'Ccdc17-SE-pos', 'Cdk5rap2-MXE-neg',
    'Col13a1-SE-neg', 'Commd1-SE-pos', 'Csde1-SE-neg', 'Dcun1d2-SE-pos',
    'Dnm1l-SE-neg', 'Etl4-SE-pos', 'Far1-MXE-pos', 'Fbf1-MXE-neg',
    'Gm16565-SE-pos', 'Gm27252-SE-neg', 'Gm46430-SE-neg', 'H2-T10-SE-pos',
    'Hdnr-SE-neg', 'Hmgn3-SE-neg', 'Hnrnpdl-SE-pos', 'Hyi-SE-pos',
    'Immt_alt-exon', 'Immt_alt-exon-junction', 'Immt_flanking-exons',
    'Kcng2-SE-pos', 'Ldb3-SE-neg', 'Ldb3_MUT-alt-exon', 'Ldb3_WT-alt-exon',
    'Ldb3_flanking-exons', 'Lrrfip2-SE-neg', 'Lyplal1-SE-neg',
    'Med15-MXE-pos', 'Mical1-ASS-pos', 'Mlip-SE-pos', 'Mov10-SE-neg',
    'Mutyh-SE-pos', 'Nup210-ASS-neg', 'Oas1c-SE-neg',
    'Pdlim5_double-junction', 'Pdlim5_flanking-exons', 'Phldb1-SE-neg',
    'Pkd2l2-SE-neg', 'Polr1b-SE-pos', 'Postn-SE-pos', 'Ppa2-SE-pos',
    'Pstk-SE-neg', 'Rbm20-P635L', 'Rbm20-R636Q', 'Rbm20-R636Q_edited',
    'Rbm20-WT', 'Rbm20_binder-sequence', 'Rbm6-SE-pos', 'Rnf44-ASS-pos',
    'Ryr2_flanking-exons', 'Sdccag8-SE-neg', 'Sel1l-ASS-neg',
    'Sema3b-SE-pos', 'Slc25a37-RI-neg', 'Smox-MXE-pos', 'Sorbs1-RI-pos',
    'Spata6-SE-neg', 'Svil-SE-neg', 'Tor1aip1-SE-pos', 'Tpcn2-SE-neg',
    'Tpm2-ASS-pos', 'Tpm2-MXE-neg', 'Tpm2-SE-pos', 'Tpm2_MT', 'Tpm2_WT',
    'Trmt1-RI-pos', 'Ttc3-SE-neg', 'Ttn-SE-pos', 'Ttn_N2A', 'Ttn_N2B',
    'Ttn_WT-N2B', 'Ttn_alt-exon', 'Ttn_flanking-exons', 'Ube2f-SE-neg',
    'Wiz-SE-neg', 'Zdhhc3-SE-neg', 'Zfp120-SE-pos', 'Zfp644-SE-neg'
]
# Exclude your custom genes from HVGs
genes_to_exclude_present = LC.var_names.intersection(exclude_genes)

# Set `LC.var.loc[genes_to_exclude_present, 'highly_variable']` for the following analysis.
LC.var.loc[genes_to_exclude_present, "highly_variable"] = False


In [ ]:
# Normalisation
sc.pp.normalize_total(LC)
# Log-transformation
sc.pp.log1p(LC)
# Make an independent copy of the selected data.
LC.layers["lognorm"] = LC.X.copy()


In [ ]:
# Scaling
sc.pp.scale(LC, max_value=10)
# Run PCA
sc.pp.pca(LC, n_comps=30)


In [ ]:
# Choose number of PCs
sc.pl.pca_variance_ratio(LC, n_pcs=30, log=True)


In [ ]:
# Run `sc.pp.neighbors` for this analysis step.
sc.pp.neighbors(LC, n_pcs = 20) #n_neighbors = 100,


In [ ]:
# Run `sc.tl.umap` for this analysis step.
sc.tl.umap(LC) #, min_dist = 0.05, spread = 2


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(LC, color = "type")


In [ ]:
# Run `sc.tl.leiden` for this analysis step.
sc.tl.leiden(LC, flavor="igraph", resolution=0.2, key_added="leiden_r02")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(LC, color = "leiden_r02")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(LC, color = ["Cd74", "Bank1", "Ms4a1", "Cd247", "Skap1", "Il2ra", "Cd4", "Ccl4", "Ccl5", "F13a1", "Mrc1", "Ccr1", "Cd68", "Cd86"])


In [ ]:
# Run `sc.tl.rank_genes_groups` for this analysis step.
sc.tl.rank_genes_groups(LC, method = "wilcoxon", groupby = "leiden_r02")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(LC, groupby = "leiden_r02", standard_scale="var", n_genes=15)


In [ ]:
# Map Leiden cluster IDs to cell type labels #Epicardium consists of the mesothelial cells
cluster_map = {
    "0": "T cells",
    "1": "T cells",
    "2": "T cells",
    "3": "B cells",
    "4": "Ccr1+ Myeloid", 
}

# Compute and store `LC.obs['celltype_LC']`.
LC.obs["celltype_LC"] = LC.obs["leiden_r02"].map(cluster_map)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(LC, color = "celltype_LC")


In [ ]:
# Save the processed object to disk.
LC.write_h5ad("Xenium_lymphoid_subset_BcellsTcells.h5ad")


In [ ]:
# Load the processed AnnData object.
LC = sc.read_h5ad("Xenium_lymphoid_subset_BcellsTcells.h5ad")


In [ ]:
# 1. Start with the original broad annotation
combined_adata.obs["celltype1"] = combined_adata.obs["celltype"].copy()

# 2. Convert to string temporarily so new LC labels can be assigned safely
combined_adata.obs["celltype1"] = combined_adata.obs["celltype1"].astype(str)

# 3. Find matching cells
shared_cells = combined_adata.obs_names.intersection(LC.obs_names)

# 4. Add detailed LC annotations
combined_adata.obs.loc[shared_cells, "celltype1"] = (
    LC.obs.loc[shared_cells, "celltype_LC"].astype(str)
)

# 5. Convert back to categorical
combined_adata.obs["celltype1"] = pd.Categorical(combined_adata.obs["celltype1"])


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype1")


In [ ]:
# Make an independent copy of the selected data.
MC = combined_adata[combined_adata.obs['celltype']=='Myeloid'].copy()


In [ ]:
# Run `sc.tl.rank_genes_groups` for this analysis step.
sc.tl.rank_genes_groups(MC, method = "wilcoxon", groupby = "type")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(MC, groupby = "type", standard_scale="var", n_genes=15)


In [ ]:
# Make an independent copy of the selected data.
MC.X = MC.layers["counts"].copy()


In [ ]:
# Run `sc.pp.highly_variable_genes` for this analysis step.
sc.pp.highly_variable_genes(MC, flavor="seurat_v3", layer="counts")


In [ ]:
# Define the values used for `exclude_genes`.
exclude_genes = [
    '1300002E11Rik-SE-neg', '2210408F21Rik-SE-pos', 'Aak1-SE-neg',
    'Abhd14a-SE-pos', 'Ank3-SE-pos', 'Arhgap10-MXE-neg', 'Arhgap10-MXE-pos',
    'Aste1-ASS-pos', 'Camk2d-SE-pos', 'Camk2d_flanking-exons',
    'Camk2d_isoformA', 'Camk2d_isoformB', 'Carnmt1-MXE-pos',
    'Ccdc125-MXE-neg', 'Ccdc17-SE-pos', 'Cdk5rap2-MXE-neg',
    'Col13a1-SE-neg', 'Commd1-SE-pos', 'Csde1-SE-neg', 'Dcun1d2-SE-pos',
    'Dnm1l-SE-neg', 'Etl4-SE-pos', 'Far1-MXE-pos', 'Fbf1-MXE-neg',
    'Gm16565-SE-pos', 'Gm27252-SE-neg', 'Gm46430-SE-neg', 'H2-T10-SE-pos',
    'Hdnr-SE-neg', 'Hmgn3-SE-neg', 'Hnrnpdl-SE-pos', 'Hyi-SE-pos',
    'Immt_alt-exon', 'Immt_alt-exon-junction', 'Immt_flanking-exons',
    'Kcng2-SE-pos', 'Ldb3-SE-neg', 'Ldb3_MUT-alt-exon', 'Ldb3_WT-alt-exon',
    'Ldb3_flanking-exons', 'Lrrfip2-SE-neg', 'Lyplal1-SE-neg',
    'Med15-MXE-pos', 'Mical1-ASS-pos', 'Mlip-SE-pos', 'Mov10-SE-neg',
    'Mutyh-SE-pos', 'Nup210-ASS-neg', 'Oas1c-SE-neg',
    'Pdlim5_double-junction', 'Pdlim5_flanking-exons', 'Phldb1-SE-neg',
    'Pkd2l2-SE-neg', 'Polr1b-SE-pos', 'Postn-SE-pos', 'Ppa2-SE-pos',
    'Pstk-SE-neg', 'Rbm20-P635L', 'Rbm20-R636Q', 'Rbm20-R636Q_edited',
    'Rbm20-WT', 'Rbm20_binder-sequence', 'Rbm6-SE-pos', 'Rnf44-ASS-pos',
    'Ryr2_flanking-exons', 'Sdccag8-SE-neg', 'Sel1l-ASS-neg',
    'Sema3b-SE-pos', 'SMC25a37-RI-neg', 'Smox-MXE-pos', 'Sorbs1-RI-pos',
    'Spata6-SE-neg', 'Svil-SE-neg', 'Tor1aip1-SE-pos', 'Tpcn2-SE-neg',
    'Tpm2-ASS-pos', 'Tpm2-MXE-neg', 'Tpm2-SE-pos', 'Tpm2_MT', 'Tpm2_WT',
    'Trmt1-RI-pos', 'Ttc3-SE-neg', 'Ttn-SE-pos', 'Ttn_N2A', 'Ttn_N2B',
    'Ttn_WT-N2B', 'Ttn_alt-exon', 'Ttn_flanking-exons', 'Ube2f-SE-neg',
    'Wiz-SE-neg', 'Zdhhc3-SE-neg', 'Zfp120-SE-pos', 'Zfp644-SE-neg'
]
# Exclude your custom genes from HVGs
genes_to_exclude_present = MC.var_names.intersection(exclude_genes)

# Set `MC.var.loc[genes_to_exclude_present, 'highly_variable']` for the following analysis.
MC.var.loc[genes_to_exclude_present, "highly_variable"] = False


In [ ]:
# Run `sc.pp.normalize_total` for this analysis step.
sc.pp.normalize_total(MC)
# Run `sc.pp.log1p` for this analysis step.
sc.pp.log1p(MC)
# Make an independent copy of the selected data.
MC.layers["lognorm"] = MC.X.copy()


In [ ]:
# Run `sc.pp.scale` for this analysis step.
sc.pp.scale(MC, max_value=10)
# Run `sc.pp.pca` for this analysis step.
sc.pp.pca(MC, n_comps=30)


In [ ]:
# Run `sc.pl.pca_variance_ratio` for this analysis step.
sc.pl.pca_variance_ratio(MC, n_pcs=30, log=True)


In [ ]:
# Run `sc.pp.neighbors` for this analysis step.
sc.pp.neighbors(MC, n_pcs = 20) #n_neighbors = 100,


In [ ]:
# Run `sc.tl.umap` for this analysis step.
sc.tl.umap(MC) #, min_dist = 0.05, spread = 2


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(MC, color = "type")


In [ ]:
# Run `sc.tl.leiden` for this analysis step.
sc.tl.leiden(MC, flavor="igraph", resolution=0.3, key_added="leidenr03")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(MC, color = "leidenr03")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(MC, color = ["Cd74", "Ccl4", "Ccl5", "F13a1", "Mrc1", "Ccr1", "Cd68", "Cd86", "Cd163", "S100a1", "Adgre1", "Ms4a7", "Lair1", "Ptprc", "Tnnt2", "Myl2"])


In [ ]:
# Run `sc.tl.rank_genes_groups` for this analysis step.
sc.tl.rank_genes_groups(MC, method = "wilcoxon", groupby = "leidenr03")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(MC, groupby = "leidenr03", standard_scale="var", n_genes=10)


In [ ]:
# Define the values used for `genes_of_interest`.
genes_of_interest = ["Cd74", "Ccl4", "Ccl5", "F13a1", "Mrc1", "Ccr1", "Cd68", "Cd86", "Cd163", "S100a1", "Adgre1", "Ms4a7", "Lair1", "Ptprc", "Col1a2", "Rbm20", "Tnnt2", "Myl2"]


# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(MC, var_names = genes_of_interest, groupby = "leidenr03", standard_scale="var")


In [ ]:
# Map Leiden cluster IDs to cell type labels #Epicardium consists of the mesothelial cells
cluster_map = {
    "0": "MCs",
    "1": "MCs",
    "2": "MC-like ECs",
    "3": "MC-like FBs",
    "4": "SMCs", 
    "5": "MCs"
}

# Compute and store `MC.obs['celltype_MC']`.
MC.obs["celltype_MC"] = MC.obs["leidenr03"].map(cluster_map)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(MC, color = "celltype_MC")


In [ ]:
# Save the processed object to disk.
MC.write_h5ad("Xenium_myeloidsubset.h5ad")


In [ ]:
# Load the processed AnnData object.
MC = sc.read_h5ad("Xenium_myeloidsubset.h5ad")


In [ ]:
# 1. Start with the original broad annotation
combined_adata.obs["celltype1"] = combined_adata.obs["celltype1"].copy()

# 2. Convert to string temporarily so new LC labels can be assigned safely
combined_adata.obs["celltype1"] = combined_adata.obs["celltype1"].astype(str)

# 3. Find matching cells
shared_cells = combined_adata.obs_names.intersection(MC.obs_names)

# 4. Add detailed LC annotations
combined_adata.obs.loc[shared_cells, "celltype1"] = (
    MC.obs.loc[shared_cells, "celltype_MC"].astype(str)
)

# 5. Convert back to categorical
combined_adata.obs["celltype1"] = pd.Categorical(combined_adata.obs["celltype1"])


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype1")


In [ ]:
# 1. Start with the original broad annotation
combined_adata.obs["celltype2"] = combined_adata.obs["celltype"].copy()

# 2. Convert to string temporarily so new LC labels can be assigned safely
combined_adata.obs["celltype2"] = combined_adata.obs["celltype2"].astype(str)

# 3. Find matching cells
shared_cells = combined_adata.obs_names.intersection(LC.obs_names)

# 4. Add detailed LC annotations
combined_adata.obs.loc[shared_cells, "celltype2"] = (
    LC.obs.loc[shared_cells, "celltype_LC"].astype(str)
)

# 5. Convert back to categorical
combined_adata.obs["celltype2"] = pd.Categorical(combined_adata.obs["celltype2"])


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype2")


In [ ]:
# Compute and store `combined_adata.obs['celltype2']`.
combined_adata.obs["celltype2"] = combined_adata.obs["celltype2"].replace({
    "Ccr1+ Myeloid": "Myeloid"
})

#remove empty categories 
combined_adata.uns.pop("celltype2_colors", None)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype2")


In [ ]:
# Define the desired order of cell types
ordered_cell_types = ['VCMs', 'Ifgga4+ VCMs','Stressed VCMs', 'Myh7+ VCMs', 'FBs', 'Vasculature ECs', 'Endocardial ECs', 'Epicardium', 'Myeloid', 'T cells', 'B cells', 'Pericytes', 'SMCs', 'NCs', 'ACMs']


In [ ]:
# Compute and store `combined_adata.obs['celltype2']`.
combined_adata.obs['celltype2'] = pd.Categorical(combined_adata.obs['celltype2'], categories=ordered_cell_types, ordered = True)


In [ ]:
# Define the values used for `set1_14`.
set1_14 = [
    "#6C5C8D", "#882255", "#24185F", "#DDCC77", "#712E00", "#88CCEE", "#97B1AB", "#44AA99", "#117733", "#999933", "#EE7733",
    "#AA4499", "#7BB4C3", "#CC6677", "#4B3FA3"

]


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype2", palette=set1_14, save = "UMAP_noWT3_withT_Bcells.pdf")


In [ ]:
# Define the desired order of cell types
ordered_types = ['WT', 'BE', 'R636Q']
# Define the values used for `ordered_names`.
ordered_names = ['WT_rep1', 'WT_rep2', 'WT_rep4', 'WT_rep5', 'BE_rep1', 'BE_rep2', 'BE_rep3', 'BE_rep4', 'PBS_rep1', 'PBS_rep2', 'PBS_rep3', 'PBS_rep4']


In [ ]:
# Compute and store `combined_adata.obs['type']`.
combined_adata.obs['type'] = pd.Categorical(combined_adata.obs['type'], categories=ordered_types, ordered = True)
# Compute and store `combined_adata.obs['name']`.
combined_adata.obs['name'] = pd.Categorical(combined_adata.obs['name'], categories=ordered_names, ordered = True)


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "celltype2"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "type"

# work from the object as-is (uses the categorical orders you already set)
obs = combined_adata.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:
group_means


## Export results

Save derived tables and figures used downstream or for manuscript preparation.


In [ ]:
# Export the resulting table as a CSV file.
group_means.to_csv("group_means_Xenium_celltypes.csv")


In [ ]:
props_sample


In [ ]:
# Export the resulting table as a CSV file.
props_sample.to_csv("samplenoWT3_Xenium_celltypes.csv")


## Combine metadata

Combine spatial measurements with cell- and sample-level annotations.


In [ ]:

# ---- Plot 1: proportions per sample ----
fig, ax = plt.subplots(figsize=(8, 5))

# Compute and store `color_dict`.
color_dict = dict(zip(celltype_order, set1_14))
# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `props_sample[plot_order].plot` for this analysis step.
props_sample[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[color_dict[ct] for ct in plot_order],
    width=0.9
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=color_dict[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_proportions_per_sample.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Display the completed figure.
plt.show()


In [ ]:
# ---- Plot 2: mean proportions per group ----
fig, ax = plt.subplots(figsize=(4.5, 5))

# Compute and store `color_dict`.
color_dict = dict(zip(celltype_order, set1_14))
# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `group_means[plot_order].plot` for this analysis step.
group_means[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[color_dict[ct] for ct in plot_order],
    width=0.8
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=color_dict[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Mean cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_mean_proportions_per_group.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Display the completed figure.
plt.show()


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad('combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells.h5ad')


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad('combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells.h5ad')


In [ ]:
# Define the values used for `celltypes_to_keep`.
celltypes_to_keep = ["VCMs", "Ifgga4+ VCMs", "Stressed VCMs", "Myh7+ VCMs"]

# Make an independent copy of the selected data.
VCMs = combined_adata[combined_adata.obs["celltype"].isin(celltypes_to_keep)].copy()


In [ ]:
# Define the values used for `genes_by_group`.
genes_by_group = {
    "VCMs": ["Ttn", "Myl2", "Tnnt2", "Ttn_N2B", "Ttn_flanking-exons", 'Camk2d_isoformA'],
    "Ifgga4+ VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons", 'Camk2d_isoformA'],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon", 'Camk2d_isoformB'],
    "Myh7+ VCMs": ["Myh7"]
}


In [ ]:
# Define the values used for `genes_by_group`.
genes_by_group = {
    "VCMs": ["Myl2", "Ttn_N2B", "Ttn_flanking-exons"],
    "Ifgga4+ VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons"],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon"],
    "Myh7+ VCMs": ["Myh7"]
}


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = 'celltype', var_names=genes_by_group, standard_scale = 'var', dendrogram=False, save = "dotplot_mygenes_grouped_VCMsspatial.pdf")


In [ ]:
# Make an independent copy of the selected data.
Ifgga4 = combined_adata[combined_adata.obs['celltype'] == 'Ifgga4+ VCMs'].copy()


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Ifgga4)


In [ ]:
# Run `sc.tl.leiden` for this analysis step.
sc.tl.leiden(Ifgga4, flavor="igraph", resolution=0.5, key_added="leidenr03")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Ifgga4, color = "leidenr03")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Ifgga4, color = "F830016B08Rik")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = ["F830016B08Rik", 'Cdh5'])


In [ ]:
# Run `sc.tl.rank_genes_groups` for this analysis step.
sc.tl.rank_genes_groups(Ifgga4, method = "wilcoxon", groupby = "leidenr03")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(Ifgga4, groupby = "leidenr03", standard_scale="var", n_genes=10)


In [ ]:
# Map Leiden cluster IDs to cell type labels #Epicardium consists of the mesothelial cells
cluster_map = {
    "0": "Ifgga4+ VCMs",
    "1": "Vasculature ECs",
    "2": "Ifgga4+ VCMs",
    "3": "VCMs"
}

# Compute and store `Ifgga4.obs['celltypeIfgga4']`.
Ifgga4.obs["celltypeIfgga4"] = Ifgga4.obs["leidenr03"].map(cluster_map)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Ifgga4, color = "celltypeIfgga4")


In [ ]:
# 1. Start with the original broad annotation
combined_adata.obs["celltype3"] = combined_adata.obs["celltype2"].copy()

# 2. Convert to string temporarily so new LC labels can be assigned safely
combined_adata.obs["celltype3"] = combined_adata.obs["celltype3"].astype(str)

# 3. Find matching cells
shared_cells = combined_adata.obs_names.intersection(Ifgga4.obs_names)

# 4. Add detailed LC annotations
combined_adata.obs.loc[shared_cells, "celltype3"] = (
    Ifgga4.obs.loc[shared_cells, "celltypeIfgga4"].astype(str)
)

# 5. Convert back to categorical
combined_adata.obs["celltype3"] = pd.Categorical(combined_adata.obs["celltype3"])


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = 'celltype3')


In [ ]:
# Make an independent copy of the selected data.
Stressed = combined_adata[combined_adata.obs['celltype'] == 'Stressed VCMs'].copy()


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Stressed)


In [ ]:
# Run `sc.tl.leiden` for this analysis step.
sc.tl.leiden(Stressed, flavor="igraph", resolution=0.8, key_added="leidenr08")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Stressed, color = "leidenr08")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Stressed, color = "F830016B08Rik")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(Stressed, color = ["F830016B08Rik", 'Cdh5'])


In [ ]:
# Run `sc.tl.rank_genes_groups` for this analysis step.
sc.tl.rank_genes_groups(Stressed, method = "wilcoxon", groupby = "leidenr08")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(Stressed, groupby = "leidenr08", standard_scale="var", n_genes=15)


In [ ]:
# Make an independent copy of the selected data.
VCMs = combined_adata[combined_adata.obs['celltype'] == 'VCMs'].copy()


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs)


In [ ]:
# Run `sc.tl.leiden` for this analysis step.
sc.tl.leiden(VCMs, flavor="igraph", resolution=0.5, key_added="leidenr05")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = "leidenr05")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = "F830016B08Rik")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = ["F830016B08Rik", 'Cdh5', 'type'])


In [ ]:
# Run `sc.tl.rank_genes_groups` for this analysis step.
sc.tl.rank_genes_groups(VCMs, method = "wilcoxon", groupby = "leidenr05")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "leidenr05", standard_scale="var", n_genes=15)


In [ ]:
# Map Leiden cluster IDs to cell type labels #Epicardium consists of the mesothelial cells
cluster_map = {
    "0": "VCMs",
    "1": "Ifgga4+ VCMs",
    "2": "VCMs",
    "3": "VCMs",
    "4": "VCMs",
    "5": "VCMs",
    "6": "VCMs"
}

# Compute and store `VCMs.obs['celltypeIfgga4']`.
VCMs.obs["celltypeIfgga4"] = VCMs.obs["leidenr05"].map(cluster_map)


In [ ]:
# 3. Find matching cells
shared_cells = combined_adata.obs_names.intersection(VCMs.obs_names)

# 4. Add detailed LC annotations
combined_adata.obs.loc[shared_cells, "celltype3"] = (
    VCMs.obs.loc[shared_cells, "celltypeIfgga4"].astype(str)
)

# 5. Convert back to categorical
combined_adata.obs["celltype3"] = pd.Categorical(combined_adata.obs["celltype3"])


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = ['celltype3', "F830016B08Rik"])


In [ ]:
# Define the desired order of cell types
ordered_cell_types = ['VCMs', 'Ifgga4+ VCMs','Stressed VCMs', 'Myh7+ VCMs', 'FBs', 'Vasculature ECs', 'Endocardial ECs', 'Epicardium', 'Myeloid', 'T cells', 'B cells', 'Pericytes', 'SMCs', 'NCs', 'ACMs']


In [ ]:
# Compute and store `combined_adata.obs['celltype3']`.
combined_adata.obs['celltype3'] = pd.Categorical(combined_adata.obs['celltype3'], categories=ordered_cell_types, ordered = True)


In [ ]:
# Define the values used for `set1_14`.
set1_14 = [
    "#6C5C8D", "#882255", "#24185F", "#DDCC77", "#712E00", "#88CCEE", "#97B1AB", "#44AA99", "#117733", "#999933", "#EE7733",
    "#AA4499", "#7BB4C3", "#CC6677", "#4B3FA3"

]


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype3", palette=set1_14, save = "UMAP_noWT3_withT_Bcells_reannotatedIfgga4.pdf")


In [ ]:
# Make an independent copy of the selected data.
BE_rep1 = combined_adata[combined_adata.obs['name']=='BE_rep1'].copy()


In [ ]:
# Plot the selected feature in spatial coordinates.
sc.pl.spatial(BE_rep1, color=["celltype3"], groups=["Ifgga4+ VCMs"], na_color = 'lightgrey', spot_size = 25)


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4.h5ad")


In [ ]:
# Define the desired order of cell types
ordered_types = ['WT', 'BE', 'R636Q']
# Define the values used for `ordered_names`.
ordered_names = ['WT_rep1', 'WT_rep2', 'WT_rep4', 'WT_rep5', 'BE_rep1', 'BE_rep2', 'BE_rep3', 'BE_rep4', 'PBS_rep1', 'PBS_rep2', 'PBS_rep3', 'PBS_rep4']


In [ ]:
# Compute and store `combined_adata.obs['type']`.
combined_adata.obs['type'] = pd.Categorical(combined_adata.obs['type'], categories=ordered_types, ordered = True)
# Compute and store `combined_adata.obs['name']`.
combined_adata.obs['name'] = pd.Categorical(combined_adata.obs['name'], categories=ordered_names, ordered = True)


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "celltype3"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "type"

# work from the object as-is (uses the categorical orders you already set)
obs = combined_adata.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:
group_means


In [ ]:
# Export the resulting table as a CSV file.
group_means.to_csv("group_means_Xenium_celltypes_Ifgga4reannotated.csv")


In [ ]:
props_sample


In [ ]:
# Export the resulting table as a CSV file.
props_sample.to_csv("samplenoWT3_Xenium_celltypes_Ifgga4reannotated.csv")


In [ ]:

# ---- Plot 1: proportions per sample ----
fig, ax = plt.subplots(figsize=(8, 5))

# Compute and store `color_dict`.
color_dict = dict(zip(celltype_order, set1_14))
# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `props_sample[plot_order].plot` for this analysis step.
props_sample[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[color_dict[ct] for ct in plot_order],
    width=0.9
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=color_dict[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_proportions_per_sample_Ifgga4reannotated.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Display the completed figure.
plt.show()


In [ ]:
# ---- Plot 2: mean proportions per group ----
fig, ax = plt.subplots(figsize=(4.5, 5))

# Compute and store `color_dict`.
color_dict = dict(zip(celltype_order, set1_14))
# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `group_means[plot_order].plot` for this analysis step.
group_means[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[color_dict[ct] for ct in plot_order],
    width=0.8
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=color_dict[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Mean cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_mean_proportions_per_group_Ifgga4reannotated.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Display the completed figure.
plt.show()


In [ ]:
# Define the values used for `celltypes_to_keep`.
celltypes_to_keep = ["VCMs", "Ifgga4+ VCMs", "Stressed VCMs", "Myh7+ VCMs"]

# Make an independent copy of the selected data.
VCMs = combined_adata[combined_adata.obs["celltype"].isin(celltypes_to_keep)].copy()


In [ ]:
# Define the values used for `genes_by_group`.
genes_by_group = {
    "VCMs": ["Ttn", "Myl2", "Tnnt2", "Ttn_N2B", "Ttn_flanking-exons", 'Camk2d_isoformA'],
    "Ifgga4+ VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons", 'Camk2d_isoformA'],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon", 'Camk2d_isoformB'],
    "Myh7+ VCMs": ["Myh7"]
}


In [ ]:
# Define the values used for `genes_by_group`.
genes_by_group = {
    "VCMs": ["Myl2", "Ttn_N2B", "Ttn_flanking-exons"],
    "Ifgga4+ VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons"],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon"],
    "Myh7+ VCMs": ["Myh7"]
}


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = 'celltype', var_names=genes_by_group, standard_scale = 'var', dendrogram=False, save = "dotplot_mygenes_grouped_VCMsspatial_Ifgga4_reannotated.pdf")


In [ ]:
# Convert values to the required data type.
combined_adata.obs["celltype4"] = combined_adata.obs["celltype3"].astype(str)

# Compute and store `combined_adata.obs['celltype4']`.
combined_adata.obs["celltype4"] = combined_adata.obs["celltype4"].replace({
    "T cells": "Lymphoid",
    "B cells": "Lymphoid"
})

# Convert values to the required data type.
combined_adata.obs["celltype4"] = combined_adata.obs["celltype4"].astype("category")


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4.h5ad")


In [ ]:
# Define the desired order of cell types
ordered_cell_types = ['VCMs', 'Ifgga4+ VCMs','Stressed VCMs', 'Myh7+ VCMs', 'FBs', 'Vasculature ECs', 'Endocardial ECs', 'Epicardium', 'Myeloid', 'Lymphoid', 'Pericytes', 'SMCs', 'NCs', 'ACMs']


In [ ]:
# Compute and store `combined_adata.obs['celltype4']`.
combined_adata.obs['celltype4'] = pd.Categorical(combined_adata.obs['celltype4'], categories=ordered_cell_types, ordered = True)


In [ ]:
# Define the values used for `set1_14`.
set1_14 = [
    "#6C5C8D", "#882255", "#24185F", "#DDCC77", "#712E00", "#88CCEE", "#97B1AB", "#44AA99", "#117733", "#999933",
    "#AA4499", "#7BB4C3", "#CC6677", "#4B3FA3"

]


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype4", palette=set1_14, save = "UMAP_noWT3_withT_Bcells_reannotatedIfgga4_noTBcells.pdf")


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4.h5ad")


In [ ]:
# Define the desired order of cell types
ordered_types = ['WT', 'BE', 'R636Q']
# Define the values used for `ordered_names`.
ordered_names = ['WT_rep1', 'WT_rep2', 'WT_rep4', 'WT_rep5', 'BE_rep1', 'BE_rep2', 'BE_rep3', 'BE_rep4', 'PBS_rep1', 'PBS_rep2', 'PBS_rep3', 'PBS_rep4']


In [ ]:
# Compute and store `combined_adata.obs['type']`.
combined_adata.obs['type'] = pd.Categorical(combined_adata.obs['type'], categories=ordered_types, ordered = True)
# Compute and store `combined_adata.obs['name']`.
combined_adata.obs['name'] = pd.Categorical(combined_adata.obs['name'], categories=ordered_names, ordered = True)


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "celltype4"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "type"

# work from the object as-is (uses the categorical orders you already set)
obs = combined_adata.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:
group_means


In [ ]:
# Export the resulting table as a CSV file.
group_means.to_csv("group_means_Xenium_celltypes_Ifgga4reannotated_noTBjustLymphoid.csv")


In [ ]:
props_sample


In [ ]:
# Export the resulting table as a CSV file.
props_sample.to_csv("samplenoWT3_Xenium_celltypes_Ifgga4reannotated_noTBjustLymphoid.csv")


In [ ]:

# ---- Plot 1: proportions per sample ----
fig, ax = plt.subplots(figsize=(8, 5))

# Compute and store `color_dict`.
color_dict = dict(zip(celltype_order, set1_14))
# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `props_sample[plot_order].plot` for this analysis step.
props_sample[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[color_dict[ct] for ct in plot_order],
    width=0.9
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=color_dict[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_proportions_per_sample_Ifgga4reannotated_noTBjustLymphoid.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Display the completed figure.
plt.show()


In [ ]:
# ---- Plot 2: mean proportions per group ----
fig, ax = plt.subplots(figsize=(4.5, 5))

# Compute and store `color_dict`.
color_dict = dict(zip(celltype_order, set1_14))
# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `group_means[plot_order].plot` for this analysis step.
group_means[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[color_dict[ct] for ct in plot_order],
    width=0.8
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=color_dict[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Mean cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_mean_proportions_per_group_Ifgga4reannotated_noTBjustLymphoid.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Display the completed figure.
plt.show()


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad("combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4.h5ad")


In [ ]:
# Make an independent copy of the selected data.
BE_rep1 = combined_adata[combined_adata.obs['name'] == 'BE_rep1'].copy()


In [ ]:
# Plot the selected feature in spatial coordinates.
sc.pl.spatial(BE_rep1, color=["celltype3"], groups=["Ifgga4+ VCMs", "VCMs", "Stressed VCMs", "Myh7+ VCMs"], na_color = 'lightgrey', spot_size = 25)


In [ ]:
# Plot the selected feature in spatial coordinates.
sc.pl.spatial(BE_rep1, color=["celltype3"], groups=["Stressed VCMs","Myh7+ VCMs"], na_color = 'lightgrey', spot_size = 25)


In [ ]:
# Plot the selected feature in spatial coordinates.
sc.pl.spatial(BE_rep1, color=["celltype3"], groups=["Myh7+ VCMs"], na_color = 'lightgrey', spot_size = 25)


In [ ]:
# Set `adata` for the following analysis.
adata = combined_adata


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "celltype4"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "type"

# work from the object as-is (uses the categorical orders you already set)
obs = combined_adata.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:
import matplotlib.pyplot as plt

# Cardiomyocyte subtypes to include
cm_celltypes = [
    "VCMs",
    "Ifgga4+ VCMs",
    "Stressed VCMs",
    "Myh7+ VCMs",
    # add any other cardiomyocyte subtypes here
]

# Keep only cell types present in props_sample
cm_celltypes = [
    ct for ct in cm_celltypes
    if ct in props_sample.columns
]

# Assign colors to the selected cardiomyocyte subtypes
color_dict = dict(zip(cm_celltypes, set1_14[:len(cm_celltypes)]))

# Select cardiomyocyte proportions
props_cm_sample = props_sample[cm_celltypes].copy()

# Rescale within each sample so cardiomyocyte subtypes sum to 100%
props_cm_sample = props_cm_sample.div(
    props_cm_sample.sum(axis=1),
    axis=0
).fillna(0)

# Reverse stacking order while keeping legend order unchanged
plot_order = cm_celltypes[::-1]

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=(8, 5))

# Run `props_cm_sample[plot_order].plot` for this analysis step.
props_cm_sample[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[color_dict[ct] for ct in plot_order],
    width=0.9
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=color_dict[ct], label=ct)
    for ct in cm_celltypes
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="Cardiomyocyte subtype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Proportion of cardiomyocytes")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Save the completed figure to disk.
fig.savefig(
    os.path.join(
        out_dir,
        "cardiomyocyte_subtype_proportions_per_sample.pdf"
    ),
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

# Display the completed figure.
plt.show()
